# Пошаговое тестирование каждой функции модуля data

## Настройка окружения


In [1]:
import sys
from pathlib import Path

# Добавляем корень проекта в PYTHONPATH
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(f"Корень проекта: {PROJECT_ROOT}")

Корень проекта: D:\Projects\PythonProject\music-genre-classification-fma


In [2]:
import pandas as pd
import numpy as np

# Настройки для отображения
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 10)

## Тестирование конфигурации и путей

In [3]:
from src.data.config import paths, audio_params, PROJECT_ROOT as SRC_ROOT

print("=== Тест 1: Конфигурация ===")
print(f"PROJECT_ROOT: {SRC_ROOT}")
print(f"\npaths.metadata_dir: {paths.metadata_dir}")
print(f"paths.active_zip: {paths.active_zip}")
print(f"paths.active_subset: {paths.active_subset}")

=== Тест 1: Конфигурация ===
PROJECT_ROOT: D:\Projects\PythonProject\music-genre-classification-fma

paths.metadata_dir: E:\music-genre-classifier\fma_metadata
paths.active_zip: E:\music-genre-classifier\fma_medium.zip
paths.active_subset: medium


In [4]:
print("=== Тест 2: Существование файлов ===")
tracks_path = paths.get_tracks_csv()
features_path = paths.get_features_csv()
genres_path = paths.get_genres_csv()

print(f"tracks.csv: {tracks_path}")
print(f"  Существует: {tracks_path.exists()}")
print(f"  Размер: {tracks_path.stat().st_size / 1024 / 1024:.2f} MB" if tracks_path.exists() else "  Файл не найден")

print(f"\nfeatures.csv: {features_path}")
print(f"  Существует: {features_path.exists()}")
print(f"  Размер: {features_path.stat().st_size / 1024 / 1024:.2f} MB" if features_path.exists() else "  Файл не найден")

print(f"\ngenres.csv: {genres_path}")
print(f"  Существует: {genres_path.exists()}")

=== Тест 2: Существование файлов ===
tracks.csv: E:\music-genre-classifier\fma_metadata\tracks.csv
  Существует: True
  Размер: 248.35 MB

features.csv: E:\music-genre-classifier\fma_metadata\features.csv
  Существует: True
  Размер: 907.06 MB

genres.csv: E:\music-genre-classifier\fma_metadata\genres.csv
  Существует: True


In [5]:
print("=== Тест 3: Аудио параметры ===")
print(f"sample_rate: {audio_params.sample_rate}")
print(f"duration: {audio_params.duration} sec")
print(f"n_mels: {audio_params.n_mels}")
print(f"n_mfcc: {audio_params.n_mfcc}")
print(f"hop_length: {audio_params.hop_length}")

=== Тест 3: Аудио параметры ===
sample_rate: 22050
duration: 30 sec
n_mels: 128
n_mfcc: 20
hop_length: 512


## Тестирование загрузчика FMALoader

In [6]:
from src.data.loader import FMALoader

print("=== Тест 4: Инициализация FMALoader ===")
loader = FMALoader()
print(f"loader.metadata_dir: {loader.metadata_dir}")

=== Тест 4: Инициализация FMALoader ===
loader.metadata_dir: E:\music-genre-classifier\fma_metadata


In [7]:
print("=== Тест 5: Загрузка tracks.csv ===")
try:
    tracks = loader.tracks
    print(f"✅ tracks загружен: {tracks.shape}")
    print(f"Индекс: {tracks.index.name}")
    print(f"Колонки (первые 5): {list(tracks.columns[:5])}")
    print(f"Тип мультииндекса: {tracks.columns.nlevels} уровня")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 5: Загрузка tracks.csv ===
Загружено tracks: (106574, 52)
✅ tracks загружен: (106574, 52)
Индекс: track_id
Колонки (первые 5): [('album', 'comments'), ('album', 'date_created'), ('album', 'date_released'), ('album', 'engineer'), ('album', 'favorites')]
Тип мультииндекса: 2 уровня


In [8]:
print("=== Тест 5.1: Структура колонок ===")
for level in tracks.columns.levels:
    print(f"  Уровень '{level}':")
    print(f"    Значения: {list(level)}")

=== Тест 5.1: Структура колонок ===
  Уровень 'Index(['album', 'artist', 'set', 'track'], dtype='str')':
    Значения: ['album', 'artist', 'set', 'track']
  Уровень 'Index(['active_year_begin', 'active_year_end', 'associated_labels', 'bio',
       'bit_rate', 'comments', 'composer', 'date_created', 'date_recorded',
       'date_released', 'duration', 'engineer', 'favorites', 'genre_top',
       'genres', 'genres_all', 'id', 'information', 'interest',
       'language_code', 'latitude', 'license', 'listens', 'location',
       'longitude', 'lyricist', 'members', 'name', 'number', 'producer',
       'publisher', 'related_projects', 'split', 'subset', 'tags', 'title',
       'tracks', 'type', 'website', 'wikipedia_page'],
      dtype='str')':
    Значения: ['active_year_begin', 'active_year_end', 'associated_labels', 'bio', 'bit_rate', 'comments', 'composer', 'date_created', 'date_recorded', 'date_released', 'duration', 'engineer', 'favorites', 'genre_top', 'genres', 'genres_all', 'id', '

In [9]:
print("=== Тест 5.2: Доступ к колонкам ===")
try:
    genre_col = tracks['track', 'genre_top']
    print(f"✅ Доступ к genre_top: первые 5 значений:\n{genre_col.head(5)}")
except KeyError as e:
    print(f"❌ Ошибка доступа: {e}")
    print("Попробуем альтернативный доступ...")
    try:
        genre_col = tracks['track']['genre_top']
        print(f"✅ Альтернативный доступ работает:\n{genre_col.head(5)}")
    except Exception as e2:
        print(f"❌ Альтернативный тоже не работает: {e2}")

=== Тест 5.2: Доступ к колонкам ===
✅ Доступ к genre_top: первые 5 значений:
track_id
2     Hip-Hop
3     Hip-Hop
5     Hip-Hop
10        Pop
20        NaN
Name: (track, genre_top), dtype: str


In [13]:
print("=== Тест 6: Загрузка features.csv ===")
try:
    features = loader.features
    print(f"✅ features загружены: {features.shape}")
    print(f"Колонки (первые 5): {list(features.columns[:5])}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 6: Загрузка features.csv ===
Загружено features: (106574, 518)
✅ features загружены: (106574, 518)
Колонки (первые 5): [('chroma_cens', 'kurtosis', '01'), ('chroma_cens', 'kurtosis', '02'), ('chroma_cens', 'kurtosis', '03'), ('chroma_cens', 'kurtosis', '04'), ('chroma_cens', 'kurtosis', '05')]


In [10]:
print("=== Тест 7: Загрузка genres.csv ===")
try:
    genres = loader.genres
    print(f"✅ genres загружены: {genres.shape}")
    print(f"genres.head():\n{genres.head()}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 7: Загрузка genres.csv ===
Загружено genres: (163, 4)
✅ genres загружены: (163, 4)
genres.head():
          #tracks  parent          title  top_level
genre_id                                           
1            8693      38    Avant-Garde         38
2            5271       0  International          2
3            1752       0          Blues          3
4            4126       0           Jazz          4
5            4106       0      Classical          5


In [11]:
print("=== Тест 8: Фильтрация по подмножеству ===")
try:
    medium_tracks = loader.get_tracks_by_subset('medium')
    print(f"✅ Треков в Medium: {len(medium_tracks)}")

    # Проверяем все ли треки из Medium
    subsets = medium_tracks['set', 'subset'].unique()
    print(f"Уникальные подмножества: {subsets}")
    assert len(subsets) == 1 and subsets[0] == 'medium', "Не все треки из Medium!"
    print("✅ Проверка пройдена: все треки из Medium")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 8: Фильтрация по подмножеству ===
Треков в medium: 25000
✅ Треков в Medium: 25000
Уникальные подмножества: <StringArray>
['small', 'medium']
Length: 2, dtype: str
❌ Ошибка: Не все треки из Medium!


In [12]:
print("=== Тест 9: Доступные подмножества ===")
try:
    all_subsets = loader.tracks['set', 'subset'].unique()
    print(f"Доступные подмножества в данных: {all_subsets}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 9: Доступные подмножества ===
Доступные подмножества в данных: <StringArray>
['small', 'medium', 'large']
Length: 3, dtype: str


In [14]:
print("=== Тест 10: Получение разбиения train/val/test ===")
try:
    splits = loader.get_available_splits(medium_tracks)
    print(f"✅ Разбиение получено:")
    print(f"  training: {len(splits['training'])} треков")
    print(f"  validation: {len(splits['validation'])} треков")
    print(f"  test: {len(splits['test'])} треков")

    total = sum(len(v) for v in splits.values())
    print(f"  Всего: {total} ({total/len(medium_tracks)*100:.1f}% от всех)")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 10: Получение разбиения train/val/test ===
✅ Разбиение получено:
  training: 19922 треков
  validation: 2505 треков
  test: 2573 треков
  Всего: 25000 (100.0% от всех)


In [15]:
print("=== Тест 11: Маппинг жанров ===")
try:
    genre_mapping = loader.get_genre_mapping()
    print(f"✅ Жанров в маппинге: {len(genre_mapping)}")
    print(f"Примеры: {dict(list(genre_mapping.items())[:5])}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 11: Маппинг жанров ===
✅ Жанров в маппинге: 163
Примеры: {1: 'Avant-Garde', 2: 'International', 3: 'Blues', 4: 'Jazz', 5: 'Classical'}


## Тестирование препроцессора

In [16]:
from src.data.preprocessor import DataPreprocessor

print("=== Тест 12: Инициализация препроцессора ===")
preprocessor = DataPreprocessor(min_samples_per_genre=100)
print(f"✅ Препроцессор создан")
print(f"  min_samples_per_genre: {preprocessor.min_samples_per_genre}")

=== Тест 12: Инициализация препроцессора ===
✅ Препроцессор создан
  min_samples_per_genre: 100


In [17]:
print("=== Тест 13: Фильтрация редких жанров ===")

genre_col = ('track', 'genre_top')
genre_counts = medium_tracks[genre_col].value_counts()
print(f"Жанров до фильтрации: {len(genre_counts)}")
print(f"Треков до фильтрации: {len(medium_tracks)}")

=== Тест 13: Фильтрация редких жанров (на маленькой выборке) ===
Жанров до фильтрации: 16
Треков до фильтрации: 25000


In [18]:
print("=== Тест 13: Фильтрация редких жанров ===")

genre_col = ('track', 'genre_top')
genre_counts = medium_tracks[genre_col].value_counts()
print(f"Жанров до фильтрации: {len(genre_counts)}")
print(f"Треков до фильтрации: {len(medium_tracks)}")

filtered_tracks = preprocessor.filter_rare_genres(medium_tracks, genre_col)
print(f"\nЖанров после фильтрации: {len(filtered_tracks[genre_col].value_counts())}")
print(f"Треков после фильтрации: {len(filtered_tracks)}")

=== Тест 13: Фильтрация редких жанров (на маленькой выборке) ===
Жанров до фильтрации: 16
Треков до фильтрации: 25000
Удалено редких жанров: 2
  - Blues: 74 треков
  - Easy Listening: 21 треков
Осталось жанров: 14
Осталось треков: 24905

Жанров после фильтрации: 14
Треков после фильтрации: 24905


In [19]:
print("=== Тест 14: Кодирование меток ===")

test_tracks = filtered_tracks.copy()
genre_col = ('track', 'genre_top')

train_mask = test_tracks['set', 'split'] == 'training'
val_mask = test_tracks['set', 'split'] == 'validation'
test_mask = test_tracks['set', 'split'] == 'test'

y_train = test_tracks[train_mask][genre_col]
y_val = test_tracks[val_mask][genre_col]
y_test = test_tracks[test_mask][genre_col]

print(f"y_train size: {len(y_train)}, y_val: {len(y_val)}, y_test: {len(y_test)}")

try:
    y_train_enc, y_val_enc, y_test_enc = preprocessor.encode_labels(y_train, y_val, y_test)
    print(f"✅ Кодирование выполнено")
    print(f"  Классов: {len(preprocessor.label_encoder.classes_)}")
    print(f"  Пример соответствия: {dict(zip(preprocessor.label_encoder.classes_, range(len(preprocessor.label_encoder.classes_))))}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 14: Кодирование меток (на маленькой выборке) ===
y_train size: 19851, y_val: 2495, y_test: 2559
Закодировано классов: 14
  0: Classical
  1: Country
  2: Electronic
  3: Experimental
  4: Folk
  5: Hip-Hop
  6: Instrumental
  7: International
  8: Jazz
  9: Old-Time / Historic
  10: Pop
  11: Rock
  12: Soul-RnB
  13: Spoken
✅ Кодирование выполнено
  Классов: 14
  Пример соответствия: {'Classical': 0, 'Country': 1, 'Electronic': 2, 'Experimental': 3, 'Folk': 4, 'Hip-Hop': 5, 'Instrumental': 6, 'International': 7, 'Jazz': 8, 'Old-Time / Historic': 9, 'Pop': 10, 'Rock': 11, 'Soul-RnB': 12, 'Spoken': 13}


In [20]:
print("=== Тест 15: Нормализация признаков ===")

common_ids = test_tracks.index.intersection(features.index)
X = features.loc[common_ids]

X_train_raw = X.loc[test_tracks[train_mask].index]
X_val_raw = X.loc[test_tracks[val_mask].index]
X_test_raw = X.loc[test_tracks[test_mask].index]

print(f"X_train shape: {X_train_raw.shape}")
print(f"X_val shape: {X_val_raw.shape}")
print(f"X_test shape: {X_test_raw.shape}")

try:
    X_train_scaled, X_val_scaled, X_test_scaled = preprocessor.normalize_features(
        X_train_raw, X_val_raw, X_test_raw
    )
    print(f"✅ Нормализация выполнена")
    print(f"  X_train: mean={X_train_scaled.mean():.4f}, std={X_train_scaled.std():.4f}")
    print(f"  X_val:   mean={X_val_scaled.mean():.4f}, std={X_val_scaled.std():.4f}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 15: Нормализация признаков ===
X_train shape: (19851, 518)
X_val shape: (2495, 518)
X_test shape: (2559, 518)
X_train: mean=0.0000, std=1.0000
X_val:   mean=0.0057, std=1.0880
X_test:  mean=-0.0070, std=0.9911
✅ Нормализация выполнена
  X_train: mean=0.0000, std=1.0000
  X_val:   mean=0.0057, std=1.0880


In [21]:
print("=== Тест 16: Веса классов ===")
try:
    weights = preprocessor.get_class_weights(y_train_enc)
    print(f"✅ Веса классов вычислены")
    print(f"  Веса: {weights}")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 16: Веса классов ===
✅ Веса классов вычислены
  Веса: {np.int64(0): np.float64(2.8645021645021647), np.int64(1): np.float64(9.985412474849095), np.int64(2): np.float64(0.2807779349363508), np.int64(3): np.float64(0.787300705957008), np.int64(4): np.float64(1.1670194003527337), np.int64(5): np.float64(0.8051837430031638), np.int64(6): np.float64(1.3568694463431306), np.int64(7): np.float64(1.741926991926992), np.int64(8): np.float64(4.633753501400561), np.int64(9): np.float64(3.47531512605042), np.int64(10): np.float64(1.500453514739229), np.int64(11): np.float64(0.2495913697286695), np.int64(12): np.float64(15.08434650455927), np.int64(13): np.float64(15.08434650455927)}


In [22]:
print("=== Тест 17: Сохранение препроцессора ===")
save_path = paths.processed_data_dir / 'test_preprocessor.pkl'
try:
    preprocessor.save(save_path)
    print(f"✅ Препроцессор сохранён: {save_path}")
    print(f"  Файл существует: {save_path.exists()}")
    print(f"  Размер: {save_path.stat().st_size} bytes")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 17: Сохранение препроцессора ===
Препроцессор сохранён: D:\Projects\PythonProject\music-genre-classification-fma\data\processed\test_preprocessor.pkl
✅ Препроцессор сохранён: D:\Projects\PythonProject\music-genre-classification-fma\data\processed\test_preprocessor.pkl
  Файл существует: True
  Размер: 13531 bytes


In [23]:
print("=== Тест 18: Загрузка препроцессора ===")
new_preprocessor = DataPreprocessor()
try:
    new_preprocessor.load(save_path)
    print(f"✅ Препроцессор загружен")
    print(f"  min_samples_per_genre: {new_preprocessor.min_samples_per_genre}")
    print(f"  classes: {list(new_preprocessor.label_encoder.classes_)[:5]}...")
except Exception as e:
    print(f"❌ Ошибка: {e}")

=== Тест 18: Загрузка препроцессора ===
Препроцессор загружен: D:\Projects\PythonProject\music-genre-classification-fma\data\processed\test_preprocessor.pkl
✅ Препроцессор загружен
  min_samples_per_genre: 100
  classes: ['Classical', 'Country', 'Electronic', 'Experimental', 'Folk']...


## Тестирование работы с ZIP

In [29]:
print("=== Тест 19: Проверка ZIP архива ===")
if paths.active_zip.exists():
    import zipfile
    try:
        with zipfile.ZipFile(paths.active_zip, 'r') as zf:
            files = zf.namelist()
            print(f"✅ ZIP открывается")
            print(f"  Файлов в архиве: {len(files)}")
            print(f"  MP3 файлов: {len([f for f in files if f.endswith('.mp3')])}")
            print(f"  Примеры: {files[:3]}")
    except Exception as e:
        print(f"❌ Ошибка: {e}")
else:
    print(f"⚠️ ZIP не найден: {paths.active_zip}")

=== Тест 19: Проверка ZIP архива ===
✅ ZIP открывается
  Файлов в архиве: 25002
  MP3 файлов: 25000
  Примеры: ['fma_medium/README.txt', 'fma_medium/checksums', 'fma_medium/000/000002.mp3']


## Итоговая проверка

In [30]:
print("\n" + "=" * 60)
print("Итоговая сводка тестирования")
print("=" * 60)

checks = [
    ("Конфигурация загружена", True),
    ("tracks.csv существует", tracks_path.exists()),
    ("features.csv существует", features_path.exists()),
    ("FMALoader работает", 'loader' in dir()),
    ("Фильтрация Medium работает", len(medium_tracks) > 0),
    ("Разбиение получено", len(splits) == 3),
    ("Препроцессор работает", preprocessor is not None),
    ("Кодирование работает", 'y_train_enc' in dir()),
    ("Нормализация работает", 'X_train_scaled' in dir()),
]

all_passed = True
for name, status in checks:
    status_str = "✅" if status else "❌"
    print(f"{status_str} {name}")
    if not status:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("✅ ВСЕ ТЕСТЫ ПРОЙДЕНЫ! Можно запускать полный пайплайн.")
else:
    print("⚠️ НЕКОТОРЫЕ ТЕСТЫ НЕ ПРОЙДЕНЫ. Проверьте ошибки выше.")
print("=" * 60)


Итоговая сводка тестирования
✅ Конфигурация загружена
✅ tracks.csv существует
✅ features.csv существует
✅ FMALoader работает
✅ Фильтрация Medium работает
✅ Разбиение получено
✅ Препроцессор работает
✅ Кодирование работает
✅ Нормализация работает

✅ ВСЕ ТЕСТЫ ПРОЙДЕНЫ! Можно запускать полный пайплайн.


In [32]:
import json

test_results = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'checks': {name: status for name, status in checks},
    'data_shapes': {
        'tracks': loader.tracks.shape if 'loader' in dir() and hasattr(loader, 'tracks') else None,
        'features': loader.features.shape if 'loader' in dir() and hasattr(loader, 'features') else None,
        'medium_tracks': len(medium_tracks) if 'medium_tracks' in dir() else None,
        'train_size': len(y_train) if 'y_train' in dir() else None,
    }
}

with open(paths.results_dir / 'test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2, default=str)

print(f"✅ Результаты тестов сохранены в {paths.results_dir / 'test_results.json'}")

✅ Результаты тестов сохранены в D:\Projects\PythonProject\music-genre-classification-fma\results\test_results.json


# ЗАПУСК ПОЛНОГО ПАЙПЛАЙНА

## Настройка окружения

In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(f"Корень проекта: {PROJECT_ROOT}")

Корень проекта: D:\Projects\PythonProject\music-genre-classification-fma


In [35]:
from src.data.pipeline import DataPipeline
from src.data.load_processed import load_data, LoadProcessedData
from src.data.config import paths, audio_params

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Все модули импортированы")

✅ Все модули импортированы


## Проверка перед запуском

In [36]:
print("=== Финальная проверка перед запуском ===")
print(f"Метаданные: {paths.metadata_dir}")
print(f"  tracks.csv: {paths.get_tracks_csv().exists()}")
print(f"  features.csv: {paths.get_features_csv().exists()}")
print(f"  genres.csv: {paths.get_genres_csv().exists()}")
print(f"\nАктивный датасет: {paths.active_subset}")
print(f"Активный ZIP: {paths.active_zip}")
print(f"  ZIP существует: {paths.active_zip.exists()}")

=== Финальная проверка перед запуском ===
Метаданные: E:\music-genre-classifier\fma_metadata
  tracks.csv: True
  features.csv: True
  genres.csv: True

Активный датасет: medium
Активный ZIP: E:\music-genre-classifier\fma_medium.zip
  ZIP существует: True


## Запуск пайплайна

In [37]:
print("\n" + "="*60)
print("ЗАПУСК ПАЙПЛАЙНА")
print("="*60)

pipeline = DataPipeline(
    subset="medium",
    min_samples_per_genre=100,
    use_features=True,
    cache_dir=paths.processed_data_dir
)

print("\nПараметры пайплайна:")
print(f"  subset: {pipeline.subset}")
print(f"  min_samples_per_genre: {pipeline.min_samples_per_genre}")
print(f"  use_features: {pipeline.use_features}")
print(f"  cache_dir: {pipeline.cache_dir}")


ЗАПУСК ПАЙПЛАЙНА

Параметры пайплайна:
  subset: medium
  min_samples_per_genre: 100
  use_features: True
  cache_dir: D:\Projects\PythonProject\music-genre-classification-fma\data\processed


In [38]:
print("\n⏳ Запуск обработки данных (может занять 2-5 минут)...")
data = pipeline.run(force_reload=False)


⏳ Запуск обработки данных (может занять 2-5 минут)...
Запуск пайплайна подготовки данных (FMA medium)
Загружено tracks: (106574, 52)
Треков в medium: 25000
Треков с жанром: 25000
Удалено редких жанров: 2
  - Blues: 74 треков
  - Easy Listening: 21 треков
Осталось жанров: 14
Осталось треков: 24905

Официальное разбиение FMA:
  Train: 19851 (79.7%)
  Val:   2495 (10.0%)
  Test:  2559 (10.3%)
Загружено features: (106574, 518)

Размеры выборок после сопоставления:
  X_train: (19851, 518)
  X_val:   (2495, 518)
  X_test:  (2559, 518)
Закодировано классов: 14
  0: Classical
  1: Country
  2: Electronic
  3: Experimental
  4: Folk
  5: Hip-Hop
  6: Instrumental
  7: International
  8: Jazz
  9: Old-Time / Historic
  10: Pop
  11: Rock
  12: Soul-RnB
  13: Spoken
X_train: mean=0.0000, std=1.0000
X_val:   mean=0.0057, std=1.0880
X_test:  mean=-0.0070, std=0.9911
Препроцессор сохранён: D:\Projects\PythonProject\music-genre-classification-fma\data\processed\preprocessor.pkl

✅ Данные сохранены в

In [39]:
pipeline.print_summary()

СВОДКА ПОДГОТОВЛЕННЫХ ДАННЫХ
Датасет:     FMA MEDIUM
Треков:      24905
Признаков:   518
Жанров:      14

Разделение (официальное FMA):
  Train:     19851 (79.7%)
  Val:       2495 (10.0%)
  Test:      2559 (10.3%)

Жанры:
  0: Classical
  1: Country
  2: Electronic
  3: Experimental
  4: Folk
  5: Hip-Hop
  6: Instrumental
  7: International
  8: Jazz
  9: Old-Time / Historic
  10: Pop
  11: Rock
  12: Soul-RnB
  13: Spoken


## Анализ полученных данных

In [40]:
print("\n=== Детальный анализ данных ===")

print(f"\nРазмеры выборок:")
print(f"  X_train: {data['X_train'].shape}")
print(f"  X_val:   {data['X_val'].shape}")
print(f"  X_test:  {data['X_test'].shape}")

print(f"\nТипы данных:")
print(f"  X_train dtype: {data['X_train'].dtype}")
print(f"  y_train dtype: {data['y_train'].dtype}")

print(f"\nСтатистика нормализации (проверка):")
print(f"  X_train mean: {data['X_train'].mean():.6f}, std: {data['X_train'].std():.6f}")
print(f"  X_val mean:   {data['X_val'].mean():.6f}, std: {data['X_val'].std():.6f}")
print(f"  X_test mean:  {data['X_test'].mean():.6f}, std: {data['X_test'].std():.6f}")


=== Детальный анализ данных ===

Размеры выборок:
  X_train: (19851, 518)
  X_val:   (2495, 518)
  X_test:  (2559, 518)

Типы данных:
  X_train dtype: float64
  y_train dtype: int64

Статистика нормализации (проверка):
  X_train mean: 0.000000, std: 1.000000
  X_val mean:   0.005729, std: 1.088015
  X_test mean:  -0.006992, std: 0.991134


In [41]:
print("\n=== Информация о метках ===")
print(f"Количество классов: {len(data['genre_names'])}")
print(f"Жанры: {data['genre_names']}")

unique, counts = np.unique(data['y_train'], return_counts=True)
print(f"\nРаспределение классов (train):")
for code, genre in zip(unique, data['genre_names']):
    count = counts[code]
    pct = count / len(data['y_train']) * 100
    bar = "█" * int(pct / 2)
    print(f"  {genre:20} {count:6} ({pct:5.1f}%) {bar}")


=== Информация о метках ===
Количество классов: 14
Жанры: ['Classical', 'Country', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Jazz', 'Old-Time / Historic', 'Pop', 'Rock', 'Soul-RnB', 'Spoken']

Распределение классов (train):
  Classical               495 (  2.5%) █
  Country                 142 (  0.7%) 
  Electronic             5050 ( 25.4%) ████████████
  Experimental           1801 (  9.1%) ████
  Folk                   1215 (  6.1%) ███
  Hip-Hop                1761 (  8.9%) ████
  Instrumental           1045 (  5.3%) ██
  International           814 (  4.1%) ██
  Jazz                    306 (  1.5%) 
  Old-Time / Historic     408 (  2.1%) █
  Pop                     945 (  4.8%) ██
  Rock                   5681 ( 28.6%) ██████████████
  Soul-RnB                 94 (  0.5%) 
  Spoken                   94 (  0.5%) 


In [42]:
print("\n=== Проверка на пропуски ===")
print(f"NaN в X_train: {np.isnan(data['X_train']).sum()}")
print(f"NaN в X_val:   {np.isnan(data['X_val']).sum()}")
print(f"NaN в X_test:  {np.isnan(data['X_test']).sum()}")
print(f"Inf в X_train: {np.isinf(data['X_train']).sum()}")


=== Проверка на пропуски ===
NaN в X_train: 0
NaN в X_val:   0
NaN в X_test:  0
Inf в X_train: 0


## Проверка кэширования

In [43]:
print("\n=== Проверка кэширования ===")

cache_files = list(paths.processed_data_dir.glob("*"))
print(f"Файлы в кэше ({paths.processed_data_dir}):")
for f in cache_files:
    size = f.stat().st_size / 1024 / 1024
    print(f"  {f.name} ({size:.2f} MB)")


=== Проверка кэширования ===
Файлы в кэше (D:\Projects\PythonProject\music-genre-classification-fma\data\processed):
  label_encoder.pkl (0.00 MB)
  metadata.json (0.03 MB)
  metadata.pkl (0.01 MB)
  pipeline_medium_min100.pkl (98.63 MB)
  preprocessor.pkl (0.01 MB)
  scaler.pkl (0.01 MB)
  test_preprocessor.pkl (0.01 MB)
  X_test.npy (10.11 MB)
  X_train.npy (78.45 MB)
  X_val.npy (9.86 MB)
  y_test.npy (0.02 MB)
  y_train.npy (0.15 MB)
  y_val.npy (0.02 MB)


In [44]:
print("\n⏳ Тестируем быструю загрузку из кэша...")
loader = LoadProcessedData(subset="medium", min_samples_per_genre=100)
if loader.exists():
    data_reloaded = loader.load()
    print(f"✅ Данные загружены из кэша")
    print(f"  X_train shape: {data_reloaded['X_train'].shape}")

    if np.allclose(data['X_train'], data_reloaded['X_train']):
        print("  ✅ Данные совпадают с оригиналом")
    else:
        print("  ⚠️ Данные НЕ совпадают!")
else:
    print("❌ Кэш не найден!")


⏳ Тестируем быструю загрузку из кэша...
Загрузка данных из D:\Projects\PythonProject\music-genre-classification-fma\data\processed...
✅ Загружены данные: 19851 train, 2495 val, 2559 test
   Жанров: 14
✅ Данные загружены из кэша
  X_train shape: (19851, 518)
  ✅ Данные совпадают с оригиналом
